In [1]:
import pandas as pd
import numpy as np
from numpy import std, absolute, mean
import seaborn as sns
import sklearn as sk
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import datetime
import tensorflow as tf
from tensorflow.python.client import device_lib #GPU Check
import tensorflow.keras #keras
from tensorflow.keras import layers
from tensorflow.keras import Sequential, Input, Model
from tensorflow.keras.layers import Dense, Dropout, Flatten, Input, Add, Activation, ZeroPadding2D
from tensorflow.keras.layers import LSTM, BatchNormalization, Conv2D, AveragePooling2D, MaxPooling2D, GlobalMaxPooling2D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint #use for early stopping
from tensorflow.keras.initializers import glorot_uniform, he_uniform #to initialize random weights for filters
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image as image_utils
from tensorflow.keras.applications.imagenet_utils import preprocess_input, decode_predictions
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras import utils
from tensorflow.keras.utils import get_file, plot_model, to_categorical, model_to_dot

In [2]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [3]:
%pwd

'/Users/melekmizher/Desktop/NLP with Disaster Tweets'

In [4]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [5]:
train.head()
test.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


In [6]:
train

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1
...,...,...,...,...,...
7608,10869,NaN,NaN,Two giant cranes holding a bridge collapse int...,1
7609,10870,NaN,NaN,@aria_ahrary @TheTawniest The out of control w...,1
7610,10871,NaN,NaN,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...,1
7611,10872,NaN,NaN,Police investigating after an e-bike collided ...,1


In [7]:
train.keyword.value_counts()

fatalities               45
deluge                   42
armageddon               42
sinking                  41
damage                   41
                         ..
forest%20fire            19
epicentre                12
threat                   11
inundation               10
radiation%20emergency     9
Name: keyword, Length: 221, dtype: int64

In [8]:
train.location.value_counts()

USA                    104
New York                71
United States           50
London                  45
Canada                  29
                      ... 
MontrÌ©al, QuÌ©bec       1
Montreal                 1
ÌÏT: 6.4682,3.18287      1
Live4Heed??              1
Lincoln                  1
Name: location, Length: 3341, dtype: int64

# Lexical Analysis

In [9]:
import re

In [10]:
def cleanText(text):
    text = re.sub(r'@[A-Za-z0-9]+', '', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'&amp;', ' and ', text)
    text = re.sub(r'https?:\/\/\S+', '', text)
    text = re.sub(r'RT[\s]+', '', text)
    text = re.sub(r"can't", 'can not', text)
    text = re.sub(r"n't", ' not', text)
    text = re.sub(r"\s'\s?|'\s", ' ', text)
    text = re.sub(r"'s", ' is', text)
    text = re.sub(r"'re", ' are', text)
    text = re.sub(r"w\/", 'with ', text)
    text = re.sub(r"'ve", ' have', text)
    text = re.sub(r"'m", ' am', text)
    text = re.sub(r"'ll", ' will', text)
    text = re.sub(r"-", '', text)
    text = re.sub(r"!", '', text)
    text = re.sub(r",", '', text)
    text = re.sub(r";", '', text)
    text = re.sub(r":", '', text)
    text = re.sub(r"_", '', text)
    text = re.sub(r"\|", '', text)
    text = re.sub(r"\@", '', text)
    text = re.sub(r"/\n", '', text)
    text = re.sub(r"\?", '', text)
    text = re.sub(r"[^A-Za-z0-9]", " ", text)
    return text

In [11]:
train['text'] = train['text'].apply(cleanText)
train['text'] = train['text'].str.lower()

In [12]:
test['text'] = test['text'].apply(cleanText)
test['text'] = test['text'].str.lower()

In [13]:
test

,id,keyword,location,text
0,0,NaN,NaN,just happened a terrible car crash
1,2,NaN,NaN,heard about earthquake is different cities sta...
2,3,NaN,NaN,there is a forest fire at spot pond geese are ...
3,9,NaN,NaN,apocalypse lighting spokane wildfires
4,11,NaN,NaN,typhoon soudelor kills 28 in china and taiwan
...,...,...,...,...
3258,10861,NaN,NaN,earthquake safety los angeles safety faste...
3259,10865,NaN,NaN,storm in ri worse than last hurricane my city...
3260,10868,NaN,NaN,green line derailment in chicago
3261,10874,NaN,NaN,meg issues hazardous weather outlook hwo


In [14]:
pd.options.display.max_rows = 500

In [15]:
#pd.describe_option('display')

In [16]:
pd.describe_option('display.max_colwidth')
pd.set_option('display.max_colwidth', 110)

display.max_colwidth : int or None
    The maximum width in characters of a column in the repr of
    a pandas data structure. When the column overflows, a "..."
    placeholder is embedded in the output. A 'None' value means unlimited.
    [default: 50] [currently: 50]


In [17]:
train.text.head(500)

0                                               our deeds are the reason of this earthquake may allah forgive us all
1                                                                             forest fire near la ronge sask  canada
2      all residents asked to shelter in place are being notified by officers  no other evacuation or shelter in ...
3                                                    13000 people receive wildfires evacuation orders in california 
4                             just got sent this photo from ruby alaska as smoke from wildfires pours into a school 
5         rockyfire update    california hwy  20 closed in both directions due to lake county fire  cafire wildfires
6                       flood disaster heavy rain causes flash flooding of streets in manitou colorado springs areas
7                                                       i am on top of the hill and i can see a fire in the woods   
8                                   there is an emergency evacua

In [18]:
from textblob import TextBlob
#Create a function to get the subjectivity
def getSubjectivity(twt):
    return TextBlob(twt).sentiment.subjectivity

In [19]:
df = train.text.apply(getSubjectivity)

In [20]:
import nltk

In [21]:
#nltk.download('punkt')
#nltk.download('wordnet')
#nltk.download('omw-1.4')
#nltk.download('stopwords')

In [22]:
train.text.head()

0                                             our deeds are the reason of this earthquake may allah forgive us all
1                                                                           forest fire near la ronge sask  canada
2    all residents asked to shelter in place are being notified by officers  no other evacuation or shelter in ...
3                                                  13000 people receive wildfires evacuation orders in california 
4                           just got sent this photo from ruby alaska as smoke from wildfires pours into a school 
Name: text, dtype: object

# Semantic Analysis - Removing Stop Words

In [23]:
from nltk.corpus import stopwords

In [24]:
from gensim.parsing.preprocessing import remove_stopwords

In [25]:
#stop_words = set(stopwords.words('english'))

"""def removeStopWords(tweet):
    filtered_sentence = []
    i = 0
    while i < len(tweet):
        for word in tweet:
            if word not in stop_words:
                filtered_sentence.append(word)
            i = i + 1
    return filtered_sentence"""

In [26]:
#train.col_lemma = train.col_lemma.apply(lambda x: [item for item in x if item not in stop_words])

In [27]:
train

,id,keyword,location,text,target
0,1,NaN,NaN,our deeds are the reason of this earthquake may allah forgive us all,1
1,4,NaN,NaN,forest fire near la ronge sask canada,1
2,5,NaN,NaN,all residents asked to shelter in place are being notified by officers no other evacuation or shelter in ...,1
3,6,NaN,NaN,13000 people receive wildfires evacuation orders in california,1
4,7,NaN,NaN,just got sent this photo from ruby alaska as smoke from wildfires pours into a school,1
...,...,...,...,...,...
7608,10869,NaN,NaN,two giant cranes holding a bridge collapse into nearby homes,1
7609,10870,NaN,NaN,ahrary the out of control wild fires in california even in the northern part of the state very troubling,1
7610,10871,NaN,NaN,m1 94 0104 utc 5km s of volcano hawaii,1
7611,10872,NaN,NaN,police investigating after an ebike collided with a car in little portugal ebike rider suffered serious n...,1


# Tokenization and Lemmatization

In [28]:
test.text

0                                                                                  just happened a terrible car crash
1                                                      heard about earthquake is different cities stay safe everyone 
2                      there is a forest fire at spot pond geese are fleeing across the street i cannot save them all
3                                                                              apocalypse lighting  spokane wildfires
4                                                                       typhoon soudelor kills 28 in china and taiwan
                                                            ...                                                      
3258                                                          earthquake safety los angeles     safety fasteners xrwn
3259    storm in ri worse than last hurricane  my city and 3others hardest hit  my yard looks like it was bombed  ...
3260                                                    

In [29]:
from nltk.tokenize import word_tokenize

train.text = train.text.apply(word_tokenize)
train.text.head()

test.text = test.text.apply(word_tokenize)
test.text.head()

0                               [our, deeds, are, the, reason, of, this, earthquake, may, allah, forgive, us, all]
1                                                                    [forest, fire, near, la, ronge, sask, canada]
2    [all, residents, asked, to, shelter, in, place, are, being, notified, by, officers, no, other, evacuation,...
3                                          [13000, people, receive, wildfires, evacuation, orders, in, california]
4           [just, got, sent, this, photo, from, ruby, alaska, as, smoke, from, wildfires, pours, into, a, school]
Name: text, dtype: object

0                                                                        [just, happened, a, terrible, car, crash]
1                                          [heard, about, earthquake, is, different, cities, stay, safe, everyone]
2    [there, is, a, forest, fire, at, spot, pond, geese, are, fleeing, across, the, street, i, can, not, save, ...
3                                                                       [apocalypse, lighting, spokane, wildfires]
4                                                           [typhoon, soudelor, kills, 28, in, china, and, taiwan]
Name: text, dtype: object

In [30]:
from nltk.stem import WordNetLemmatizer

wnl = WordNetLemmatizer()

def lemmatize(s):
    s = [wnl.lemmatize(word) for word in s]
    return s

train = train.assign(text = train.text.apply(lambda x: lemmatize(x)))
train.text = train.text.squeeze()

test = test.assign(text = test.text.apply(lambda x: lemmatize(x)))
test.text = test.text.squeeze()

In [31]:
train.text.head(10)

0                                 [our, deed, are, the, reason, of, this, earthquake, may, allah, forgive, u, all]
1                                                                    [forest, fire, near, la, ronge, sask, canada]
2    [all, resident, asked, to, shelter, in, place, are, being, notified, by, officer, no, other, evacuation, o...
3                                            [13000, people, receive, wildfire, evacuation, order, in, california]
4             [just, got, sent, this, photo, from, ruby, alaska, a, smoke, from, wildfire, pours, into, a, school]
5    [rockyfire, update, california, hwy, 20, closed, in, both, direction, due, to, lake, county, fire, cafire,...
6          [flood, disaster, heavy, rain, cause, flash, flooding, of, street, in, manitou, colorado, spring, area]
7                                        [i, am, on, top, of, the, hill, and, i, can, see, a, fire, in, the, wood]
8                   [there, is, an, emergency, evacuation, happening, now, in, t

In [32]:
stop_words = stopwords.words('english')
stop_words.remove('not')

train.text = train.text.apply(lambda words: [word for word in words if word not in stop_words])

test.text = test.text.apply(lambda words: [word for word in words if word not in stop_words])

In [33]:
train.head(5)
test.head(5)

,id,keyword,location,text,target
0,1,NaN,NaN,"[deed, reason, earthquake, may, allah, forgive, u]",1
1,4,NaN,NaN,"[forest, fire, near, la, ronge, sask, canada]",1
2,5,NaN,NaN,"[resident, asked, shelter, place, notified, officer, evacuation, shelter, place, order, expected]",1
3,6,NaN,NaN,"[13000, people, receive, wildfire, evacuation, order, california]",1
4,7,NaN,NaN,"[got, sent, photo, ruby, alaska, smoke, wildfire, pours, school]",1


,id,keyword,location,text
0,0,NaN,NaN,"[happened, terrible, car, crash]"
1,2,NaN,NaN,"[heard, earthquake, different, city, stay, safe, everyone]"
2,3,NaN,NaN,"[forest, fire, spot, pond, goose, fleeing, across, street, not, save]"
3,9,NaN,NaN,"[apocalypse, lighting, spokane, wildfire]"
4,11,NaN,NaN,"[typhoon, soudelor, kill, 28, china, taiwan]"


# Full Cross-Validation Format Setup

In [34]:
import spacy, torch, transformers
from transformers import BertModel, BertTokenizer, TFBertModel, TFBertForSequenceClassification, BertConfig, AdamW, get_linear_schedule_with_warmup

In [35]:
train.text = train.text.apply(lambda x : " ".join(x))

In [36]:
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=1000, split=' ') 
tokenizer.fit_on_texts(train['text'].values)
X_train = tokenizer.texts_to_sequences(train['text'].values)
X_train = pad_sequences(X_train, maxlen=15)

In [37]:
X_test = tokenizer.texts_to_sequences(test['text'].values)
X_test = pad_sequences(X_test, maxlen=15)

In [38]:
pd.DataFrame(X_train).head(5)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,0,0,0,0,0,0,0,0,0,0,0,462,165,67,6
1,0,0,0,0,0,0,0,0,0,0,0,112,3,153,589
2,0,0,0,0,0,0,0,0,0,463,332,166,463,333,964
3,0,0,0,0,0,0,0,0,0,0,12,79,166,333,33
4,0,0,0,0,0,0,0,0,0,0,31,110,190,79,104


In [39]:
pd.DataFrame(X_test).head(5)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,0,0,0,0,0,0,0,0,0,0,0,0,777,46,32
1,0,0,0,0,0,0,0,0,0,0,388,165,105,433,146
2,0,0,0,0,0,0,0,0,112,3,638,736,449,1,259
3,0,0,0,0,0,0,0,0,0,0,0,0,0,390,79
4,0,0,0,0,0,0,0,0,0,0,0,414,583,108,252


# #1 LSTM Build.

In [40]:
from keras.layers import Dense, Embedding, LSTM, SpatialDropout1D

model = Sequential()
model.add(Embedding(500, 120, input_length = X_train.shape[1]))
model.add(SpatialDropout1D(0.4))
model.add(LSTM(176, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss = 'binary_crossentropy', optimizer='adam', metrics = ['accuracy'])
print(model.summary())

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 15, 120)           60000     
                                                                 
 spatial_dropout1d (SpatialD  (None, 15, 120)          0         
 ropout1D)                                                       
                                                                 
 lstm (LSTM)                 (None, 176)               209088    
                                                                 
 dense (Dense)               (None, 1)                 177       
                                                                 
Total params: 269,265
Trainable params: 269,265
Non-trainable params: 0
_________________________________________________________________
None


2022-05-30 00:31:03.690562: I tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [41]:
from sklearn.model_selection import train_test_split

train_valid, test_valid = train_test_split(X_train, random_state=42, train_size=0.8)
print(train_valid.shape)
print(test_valid.shape)

(6090, 15)
(1523, 15)


In [42]:
y_train_valid, y_test_valid = train_test_split(train.target, random_state=42, train_size=0.8)

In [43]:
train_valid[0:5]
test_valid[0:5]
y_train_valid.head(5)
y_test_valid.head(5)

array([[  0,   0,   0,   0,   0,   0,   0,   0,   0,  64, 402, 144,  36,
        987, 177],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 368,
        380, 145],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0, 491,   6, 382, 203,
        291, 618],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 341, 308, 649,
        175, 111],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 420, 407,
        371, 338]], dtype=int32)

array([[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   8,  72,
        130, 303],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,  92,  31,
        349,  53],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,  22],
       [  0,   0,   0, 740,  34, 104,   2, 127,  62, 425, 146, 515, 450,
        127, 172],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 302, 201, 718,
         10,  47]], dtype=int32)

4996    1
3263    0
4907    1
2855    1
4716    0
Name: target, dtype: int64

2644    1
2227    0
5448    1
132     0
6845    0
Name: target, dtype: int64

In [44]:
batch_size=10
model.fit(train_valid, y_train_valid, epochs = 10, batch_size=batch_size, validation_data =(test_valid, y_test_valid) , verbose = 'auto')

Epoch 1/10


InvalidArgumentError:  indices[0,6] = 906 is not in [0, 500)
	 [[node sequential/embedding/embedding_lookup
 (defined at /opt/anaconda3/envs/tensorflow/lib/python3.10/site-packages/keras/layers/embeddings.py:189)
]] [Op:__inference_train_function_3027]

Errors may have originated from an input operation.
Input Source operations connected to node sequential/embedding/embedding_lookup:
In[0] sequential/embedding/embedding_lookup/1768:	
In[1] sequential/embedding/Cast:

Operation defined at: (most recent call last)
>>>   File "/opt/anaconda3/envs/tensorflow/lib/python3.10/runpy.py", line 191, in _run_module_as_main
>>>     msg = "%s: %s" % (sys.executable, exc)
>>> 
>>>   File "/opt/anaconda3/envs/tensorflow/lib/python3.10/runpy.py", line 75, in _run_code
>>>     fname = mod_spec.origin
>>> 
>>>   File "/opt/anaconda3/envs/tensorflow/lib/python3.10/site-packages/ipykernel_launcher.py", line 12, in <module>
>>>     if sys.path[0] == "":
>>> 
>>>   File "/opt/anaconda3/envs/tensorflow/lib/p

In [ ]:
y_pred = model.predict(X_test)

threshold =0.5
y_pred = np.where(y_pred > threshold, 1, 0)
y_pred = pd.DataFrame(y_pred)
y_pred.columns = ['target']

In [ ]:
y_pred.head(100)

In [ ]:
submission = pd.DataFrame(test.id)

In [ ]:
submission['target'] = y_pred

In [ ]:
submission = submission.set_index('id')

In [ ]:
submission.target.value_counts()

In [ ]:
submission.head(10)

In [ ]:
submission.to_csv("submission.csv")

# First Submission Kaggle Score: 0.75543

# #2 LSTM Build.

In [45]:
max_features = 3000
embed_dim = 32
lstm_out = 32
model = Sequential()
model.add(Embedding(max_features, embed_dim,input_length = X_train.shape[1]))
model.add(Dropout(0.2))
model.add(LSTM(lstm_out, dropout=0.2, recurrent_dropout=0.4))
model.add(Dense(1,activation='sigmoid'))
adam = tensorflow.keras.optimizers.Adam(learning_rate=0.002)
model.compile(loss = 'binary_crossentropy', optimizer=adam ,metrics = ['accuracy'])
print(model.summary())

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 15, 32)            96000     
                                                                 
 dropout (Dropout)           (None, 15, 32)            0         
                                                                 
 lstm_1 (LSTM)               (None, 32)                8320      
                                                                 
 dense_1 (Dense)             (None, 1)                 33        
                                                                 
Total params: 104,353
Trainable params: 104,353
Non-trainable params: 0
_________________________________________________________________
None


In [46]:
batch_size=64
model.fit(train_valid, y_train_valid, epochs = 5, batch_size=batch_size, validation_data =(test_valid, y_test_valid) , verbose = 'auto')

Epoch 1/5
96/96 [==============================] - 3s 17ms/step - loss: 0.5787 - accuracy: 0.6877 - val_loss: 0.4881 - val_accuracy: 0.7754
Epoch 2/5
96/96 [==============================] - 1s 13ms/step - loss: 0.4145 - accuracy: 0.8153 - val_loss: 0.4780 - val_accuracy: 0.7708
Epoch 3/5
96/96 [==============================] - 1s 14ms/step - loss: 0.3878 - accuracy: 0.8312 - val_loss: 0.4898 - val_accuracy: 0.7695
Epoch 4/5
96/96 [==============================] - 1s 14ms/step - loss: 0.3743 - accuracy: 0.8345 - val_loss: 0.5072 - val_accuracy: 0.7682
Epoch 5/5
96/96 [==============================] - 1s 14ms/step - loss: 0.3654 - accuracy: 0.8422 - val_loss: 0.5084 - val_accuracy: 0.7728


In [47]:
y_pred = model.predict(X_test)

threshold =0.5
y_pred = np.where(y_pred > threshold, 1, 0)
y_pred = pd.DataFrame(y_pred)
y_pred.columns = ['target']
submission = pd.DataFrame(test.id)
submission['target'] = y_pred
submission = submission.set_index('id')
submission.target.value_counts()

0    1981
1    1282
Name: target, dtype: int64

In [48]:
submission.to_csv("submission.csv")

# Second Submission Kaggle Score: 0.77290

# #3 LSTM Build

In [ ]:
max_features = 800
embed_dim = 64
lstm_out = 32
model = Sequential()
model.add(Embedding(max_features, embed_dim,input_length = X_train.shape[1]))
model.add(Dropout(0.2))
model.add(LSTM(lstm_out, dropout=0.2, recurrent_dropout=0.4))
model.add(Dense(1,activation='sigmoid'))
adam = tensorflow.keras.optimizers.Adam(learning_rate=0.002)
model.compile(loss = 'binary_crossentropy', optimizer=adam ,metrics = ['accuracy'])
print(model.summary())

In [ ]:
batch_size=16
model.fit(train_valid, y_train_valid, epochs = 3, batch_size=batch_size, validation_data =(test_valid, y_test_valid) , verbose = 'auto')

In [ ]:
y_pred = model.predict(X_test)

threshold =0.5
y_pred = np.where(y_pred > threshold, 1, 0)
y_pred = pd.DataFrame(y_pred)
y_pred.columns = ['target']
submission = pd.DataFrame(test.id)
submission['target'] = y_pred
submission = submission.set_index('id')
submission.target.value_counts()

In [ ]:
submission.to_csv("submission.csv")

# Third Submission Kaggle Score: 0.77137